In [ ]:
# ============================================================
# EMNLP 2026 — "Doctrine Blindness in Legal AI"
# TASK C: Demographic × Doctrine Interaction
# COMPLETE REWRITE — fixes case_id mismatch bug
# ============================================================
# HOW TO USE:
#   Run ALL cells top to bottom in order.
#   Cell 7 prints pair counts + cost — confirm before Cell 12.
#   If Colab disconnects: re-run ALL cells from top.
#   The resume logic will skip already-done work.
#   DO NOT run individual cells out of order.
# ============================================================

# ══════════════════════════════════════════════════════════════
# CELL 1 — Install packages
# ══════════════════════════════════════════════════════════════
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "openai", "groq", "requests", "scipy", "numpy",
                "pandas", "tqdm"], check=True)
print("✓ All packages installed")

# ══════════════════════════════════════════════════════════════
# CELL 2 — Mount Google Drive
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/BailBench_Doctrine'  # update to your Drive path
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✓ Drive mounted | Results → {DRIVE_DIR}")

# ══════════════════════════════════════════════════════════════
# CELL 3 — Load API keys
# ══════════════════════════════════════════════════════════════
from google.colab import userdata

def get_secret(name):
    try:
        val = userdata.get(name)
        return val if val else None
    except Exception:
        return None

OPENAI_KEY   = get_secret('OPENAI_API_KEY')
GROQ_KEYS    = [get_secret(k) for k in [
                    'GROQ_API_KEY',   'GROQ_API_KEY_2', 'GROQ_API_KEY_3',
                    'GROQ_API_KEY_4', 'GROQ_API_KEY_5', 'GROQ_API_KEY_6',
                    'GROQ_API_KEY_7', 'GROQ_API_KEY_8']]
GROQ_KEYS    = [k for k in GROQ_KEYS if k]
MISTRAL_KEYS = [get_secret(k) for k in [
                    'MISTRAL_API_KEY', 'MISTRAL_API_KEY_2', 'MISTRAL_API_KEY_3', 'MISTRAL_API_KEY_4', 'MISTRAL_API_KEY_5']]
MISTRAL_KEYS = [k for k in MISTRAL_KEYS if k]

print(f"OpenAI  key  : {'✓ loaded' if OPENAI_KEY else '✗ MISSING'}")
print(f"Groq    keys : {len(GROQ_KEYS)} loaded")
print(f"Mistral keys : {len(MISTRAL_KEYS)} loaded")

assert OPENAI_KEY,        "❌ Add OPENAI_API_KEY to Colab Secrets"
assert len(GROQ_KEYS),    "❌ Add at least GROQ_API_KEY to Colab Secrets"
assert len(MISTRAL_KEYS), "❌ Add MISTRAL_API_KEY to Colab Secrets"
print("✓ All API keys loaded")

# ══════════════════════════════════════════════════════════════
# CELL 4 — Imports and constants
# ══════════════════════════════════════════════════════════════
import json, time, random, itertools, warnings
import pandas as pd
import numpy as np
from scipy.stats import norm, binomtest
from tqdm.auto import tqdm
from datetime import datetime
from openai import OpenAI
from groq import Groq
import requests

warnings.filterwarnings('ignore')

SEED       = 42
SAVE_EVERY = 20

random.seed(SEED)
np.random.seed(SEED)

_groq_cycle    = itertools.cycle(range(len(GROQ_KEYS)))
_mistral_cycle = itertools.cycle(range(len(MISTRAL_KEYS)))
_oa_client     = OpenAI(api_key=OPENAI_KEY)

# ── case_id normalisation — THE FIX ──────────────────────────
# pairs_df stores case_id as zero-padded 4-digit string: '0003','0007'
# partial CSV stores case_id as integer when read back by pandas: 3, 7
# This caused ALL resume keys to mismatch on every restart.
# Solution: always normalise case_id to int then to plain string: '3','7'
# Applied consistently in pairs_df, runner, and partial file loading.

def norm_cid(x):
    """Normalise case_id to plain integer string: '3', '14', '607'"""
    return str(int(str(x).lstrip('0') or '0'))

print(f"✓ Imports done | SEED={SEED} | SAVE_EVERY={SAVE_EVERY}")
print(f"  case_id normalisation: '0007' → '7', 7 → '7'")

# ══════════════════════════════════════════════════════════════
# CELL 5 — Load dataset
# ══════════════════════════════════════════════════════════════
DATASET_PATH = '/content/indian_bail_judgments.json'

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        "\n❌ Upload indian_bail_judgments.json via Colab sidebar first")

with open(DATASET_PATH) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['parity_argument_used'] = df['parity_argument_used'].astype(bool)

assert len(df) == 1200
assert df['parity_argument_used'].sum() == 341

parity_df = df[df['parity_argument_used'] == True].copy().reset_index(drop=True)
# Normalise case_id in source dataset
parity_df['case_id'] = parity_df['case_id'].apply(norm_cid)

print(f"✓ Dataset loaded: {len(parity_df)} parity=True cases")
print(f"  case_id sample: {parity_df['case_id'].head(5).tolist()}")

# ══════════════════════════════════════════════════════════════
# CELL 6 — Name swap dictionaries (complete extended set)
# ══════════════════════════════════════════════════════════════
HINDU_MARKERS = {
    'sharma','singh','kumar','gupta','verma','patel','shah','yadav',
    'mishra','tiwari','pandey','dubey','rajput','joshi','mehta',
    'ram','lal','das','mandal','thakur','choudhary','chaudhary','chauhan',
    'paswan','lodhi','khandelwal','sahani','sahni','pathak','narayan',
    'narayana','jain','bansal','agarwal','naik','pillai','roy','turi',
    'ganjhu','sawant','salve','kewat','patra','jatav','mahto','kaushik',
    'tyagi','chaubey','vaishnav','panwar','natarajan','nagaraja','manjunath',
    'rakheja','shankar','barne','dhankar','bhakat','devgan','shrivas',
    'maniar','dike','rabari','channabasappa','basappa','prajapati',
    'alagharu','rajansri','keshava','revansiddappa','chitravel','udhav',
    'ayyappa','manikantha','yogaraja','lama','kaur','ranjit','shinderpal',
    'deepak','santosh','rajesh','prakash','suresh','ramesh','mahesh',
    'ganesh','rakesh','ajay','amit','rahul','sunil','subhash','vijay',
    'ravi','ashok','avinash','rajiv','bablu','babu','guddu','pappu',
    'chandan','shambhu','kalu','raju','mohan','sanjay','dinesh','mukesh',
    'naresh','manish','satish','harish','girish','nikhil','ankit','mohit',
    'kundan','dhananjay','abhinav','pintu','rinku','mintu','golu','banti',
    'rohan','devi','chandra','rai','soni','prasad','gautam','shivam',
    'vikash','vikram','ranjan','praveen','neeraj','mahendra','kishore',
    'madan','braja','shashikant','naveen','dilip','tarun','bharath',
    'bharat','anand','anita','bindu','suman','hemu','birju','vicky',
    'dablu','ankur','rakshpal','pinki','madhuri','usha','saroj','radhe',
    'radheshyam','sunita','jyothin','narasing','kirthiraj',
}

MUSLIM_MARKERS = {
    'khan','mohammed','mohammad','shaikh','sheikh','ansari','qureshi',
    'siddiqui','ali','hussain','hassan','malik','pasha','mirza','beg',
    'ahmed','ahmad','abdul','syed','akhtar','usman','fakir','pathan',
    'haque','najmul','kalamuddin','khatoon','begum','imran','javed',
    'tanveer','zakir','naseer','wahid','hasan','umar','omar','mian',
    'mohd','md','aslam','salim','salma','amaluddin','javedkhan','jahidkhan',
    'azizkhan','alisaheb','altaf','naushad','amjad','rohim','amir',
    'ekramul','shaik','sayyed','alam','sattar','mehtab','ghanchi',
    'irfan','amzad','imrat',
}

assert len(HINDU_MARKERS & MUSLIM_MARKERS) == 0, "Marker overlap!"

H2M_SWAPS = {
    'sharma':'khan',        'singh':'ansari',       'kumar':'mohammed',
    'gupta':'siddiqui',     'verma':'qureshi',      'patel':'shaikh',
    'shah':'mirza',         'yadav':'hussain',       'mishra':'malik',
    'tiwari':'ahmed',       'pandey':'pasha',        'dubey':'beg',
    'rajput':'khan',        'joshi':'ansari',        'mehta':'siddiqui',
    'ram':'ali',            'lal':'hassan',          'das':'ahmad',
    'mandal':'ansari',      'thakur':'khan',         'choudhary':'qureshi',
    'chaudhary':'qureshi',  'chauhan':'pathan',      'paswan':'ansari',
    'lodhi':'pathan',       'khandelwal':'siddiqui', 'sahani':'ansari',
    'sahni':'ansari',       'pathak':'khan',         'narayan':'hussain',
    'narayana':'hussain',   'jain':'siddiqui',       'bansal':'ansari',
    'agarwal':'qureshi',    'naik':'shaikh',         'pillai':'ansari',
    'roy':'ahmed',          'turi':'ansari',         'ganjhu':'ansari',
    'sawant':'shaikh',      'salve':'ansari',        'kewat':'ansari',
    'patra':'pathan',       'jatav':'javed',         'mahto':'mohsin',
    'kaushik':'khalid',     'tyagi':'tahir',         'chaubey':'chand',
    'vaishnav':'wahid',     'panwar':'pathan',       'natarajan':'naseer',
    'nagaraja':'nasir',     'manjunath':'majid',     'shankar':'shakil',
    'barne':'baig',         'dhankar':'danish',      'bhakat':'bilal',
    'devgan':'danish',      'shrivas':'shaikh',      'maniar':'malik',
    'dike':'danish',        'rabari':'rashid',       'prajapati':'pathan',
    'kaur':'khatoon',       'ranjit':'rashid',       'shinderpal':'shahzad',
    'deepak':'imran',       'santosh':'saleem',      'rajesh':'rashid',
    'prakash':'farhan',     'suresh':'salman',       'ramesh':'rahim',
    'mahesh':'majid',       'ganesh':'ghulam',       'rakesh':'rehan',
    'ajay':'ajmal',         'amit':'aamir',          'rahul':'rashid',
    'sunil':'suleman',      'subhash':'sabir',       'vijay':'wasim',
    'ravi':'raza',          'ashok':'asif',          'avinash':'anwar',
    'rajiv':'razib',        'bablu':'babul',         'babu':'babar',
    'guddu':'ghulam',       'pappu':'parvez',        'chandan':'chand',
    'shambhu':'shamim',     'kalu':'khalid',         'raju':'raza',
    'mohan':'mohsin',       'sanjay':'sajid',        'dinesh':'danish',
    'mukesh':'mujib',       'naresh':'naseer',       'manish':'mannan',
    'satish':'sadiq',       'harish':'haroon',       'girish':'ghulam',
    'nikhil':'naved',       'ankit':'anas',          'mohit':'mohsin',
    'kundan':'kamil',       'dhananjay':'danish',    'abhinav':'anwar',
    'pintu':'pervez',       'rinku':'rizwan',        'golu':'ghulam',
    'banti':'bilal',        'rohan':'rehan',         'devi':'begum',
    'chandra':'chand',      'rai':'khan',            'soni':'ansari',
    'prasad':'hussain',     'gautam':'ghulam',       'shivam':'shahid',
    'vikash':'wasim',       'vikram':'imran',        'ranjan':'rizwan',
    'praveen':'pervez',     'neeraj':'naved',        'mahendra':'majid',
    'kishore':'khalid',     'madan':'moeen',         'braja':'bilal',
    'shashikant':'shahbaz', 'naveen':'naseem',       'dilip':'danish',
    'tarun':'tanveer',      'bharath':'babar',       'bharat':'babar',
    'anand':'arshad',       'anita':'amina',         'bindu':'bilqis',
    'suman':'salma',        'hemu':'hamid',          'birju':'bilal',
    'vicky':'wahid',        'dablu':'danish',        'ankur':'anas',
    'rakshpal':'rashid',    'pinki':'parveen',       'keshava':'khalid',
    'madhuri':'mahira',     'usha':'uzma',           'udhav':'usman',
    'saroj':'salma',        'ayyappa':'ayaz',        'radheshyam':'rahim',
    'radhe':'rashid',       'yogaraja':'yusuf',      'sunita':'salma',
    'jyothin':'javed',      'narasing':'nasir',      'kirthiraj':'khalid',
}

M2H_SWAPS = {
    'khan':'sharma',        'ansari':'singh',        'mohammed':'kumar',
    'siddiqui':'gupta',     'qureshi':'verma',       'shaikh':'patel',
    'mirza':'shah',         'hussain':'yadav',        'malik':'mishra',
    'ahmed':'tiwari',       'pasha':'pandey',         'beg':'dubey',
    'ali':'ram',            'hassan':'lal',           'ahmad':'das',
    'pathan':'chauhan',     'syed':'sharma',          'akhtar':'santosh',
    'usman':'suresh',       'fakir':'ramesh',         'haque':'prakash',
    'najmul':'naresh',      'kalamuddin':'krishna',   'khatoon':'kavita',
    'begum':'bharati',      'imran':'rohit',          'javed':'rajiv',
    'tanveer':'sanjay',     'zakir':'vikram',         'naseer':'naresh',
    'wahid':'vivek',        'hasan':'harish',         'umar':'umesh',
    'omar':'om',            'mian':'mohan',           'mohd':'mohan',
    'md':'mahesh',          'abdul':'rajesh',         'aslam':'ashok',
    'salim':'suresh',       'salma':'savita',         'amaluddin':'anand',
    'altaf':'ajay',         'naushad':'naresh',       'amjad':'amit',
    'rohim':'rohit',        'amir':'amit',            'shaik':'shankar',
    'sayyed':'sanjay',      'alam':'alok',            'sattar':'satish',
    'mehtab':'mahesh',      'irfan':'ishaan',         'ghanchi':'ganesh',
    'amzad':'anand',        'imrat':'ishaan',         'ghulam':'ganesh',
    'rashid':'rajesh',      'rahim':'ramesh',         'majid':'mahesh',
    'rehan':'rohan',        'ajmal':'ajay',           'aamir':'amit',
    'suleman':'sunil',      'sabir':'subhash',        'wasim':'vijay',
    'raza':'ravi',          'asif':'ashok',           'anwar':'avinash',
    'razib':'rajiv',        'shamim':'shambhu',       'khalid':'kailash',
    'sajid':'sanjay',       'danish':'dinesh',        'mujib':'mukesh',
    'mannan':'manish',      'sadiq':'satish',         'haroon':'harish',
    'naved':'naresh',       'anas':'ankit',           'mohsin':'mohan',
    'kamil':'kundan',       'bilal':'bhuvan',         'pervez':'pankaj',
    'rizwan':'rituraj',     'parvez':'pankaj',        'chand':'chandan',
    'farhan':'pankaj',      'salman':'santosh',       'babul':'bablu',
    'babar':'babu',         'parveen':'praveen',      'arshad':'aravind',
    'amina':'anita',        'bilqis':'bindu',         'uzma':'usha',
    'mahira':'madhuri',     'ayaz':'ayush',           'naseem':'naveen',
    'shahbaz':'shashikant', 'moeen':'mohan',          'tahir':'tarun',
    'shahzad':'shashank',   'hamid':'harish',         'naved':'naresh',
    'shahid':'satish',      'nasir':'naresh',         'shakil':'shankar',
}

MALE_TO_FEMALE = {
    'ramesh':'rekha',       'suresh':'sunita',        'mahesh':'mamta',
    'rajesh':'ranjana',     'mukesh':'mukta',         'dinesh':'deepa',
    'naresh':'nandita',     'umesh':'uma',            'ganesh':'gita',
    'rakesh':'rakhee',      'hitesh':'hita',          'ritesh':'rita',
    'salman':'salma',       'imran':'imrana',         'arif':'arifa',
    'mohsin':'mohsina',     'wasim':'wasima',         'irfan':'irfana',
    'bablu':'babli',        'pappu':'pappi',          'rinku':'rinki',
    'sanjay':'sangeeta',    'vijay':'vidya',          'ajay':'asha',
    'anil':'anita',         'sunil':'sunita',         'rahul':'rekha',
    'amit':'amita',         'sumit':'sumita',         'rohit':'rohini',
    'vikas':'vimala',       'deepak':'deepa',         'vinod':'vinodini',
    'ravi':'raveena',       'karan':'kiran',          'arjun':'anjali',
    'mohan':'mohini',       'rohan':'rohini',         'sohan':'sohini',
    'santosh':'sangeeta',   'ashok':'asha',           'prakash':'prabha',
    'rajiv':'rajni',        'subhash':'subhadra',     'manish':'manisha',
    'satish':'savita',      'harish':'harsha',        'girish':'girija',
    'nikhil':'nikita',      'ankit':'ankita',         'mohit':'mohita',
    'chandan':'chandana',   'kundan':'kumkum',        'abhinav':'abha',
    'pintu':'pinky',        'babu':'babita',          'guddu':'guddi',
    'raju':'raji',          'kalu':'kali',            'shambhu':'shanti',
    'avinash':'avantika',   'dhananjay':'dhara',      'naveen':'nandini',
    'bablu':'babita',       'dilip':'deepa',          'tarun':'taruna',
    'vikram':'vikrama',     'vikash':'vidya',         'praveen':'pravita',
    'neeraj':'neeraja',     'mahendra':'maheshwari',  'kishore':'kishori',
    'madan':'madhumati',    'shashikant':'shashikala','shankar':'shankari',
    'gautam':'gauri',       'rajesh':'rajeshwari',    'naresh':'nandini',
    'golu':'golu',          'mintu':'minty',
}

def is_hindu(name):
    n = str(name or '').lower()
    words = set(w.strip('.,@#()-/\\') for w in n.split())
    return bool(words & HINDU_MARKERS) and not bool(words & MUSLIM_MARKERS)

def is_muslim(name):
    n = str(name or '').lower()
    words = set(w.strip('.,@#()-/\\') for w in n.split())
    return bool(words & MUSLIM_MARKERS) and not bool(words & HINDU_MARKERS)

def do_name_swap(name, swap_dict):
    if not name or str(name).strip() in ('Unknown','Not specified',''):
        return None
    parts = str(name).split()
    swapped = []
    changed = False
    for part in parts:
        clean = part.strip('.,@#()-/\\')
        if clean.lower() in swap_dict:
            suffix = part[len(clean):]
            swapped.append(swap_dict[clean.lower()].capitalize() + suffix)
            changed = True
        else:
            swapped.append(part)
    return ' '.join(swapped) if changed else None

print("✓ Name swap dictionaries loaded")
print(f"  Hindu markers: {len(HINDU_MARKERS)} | Muslim markers: {len(MUSLIM_MARKERS)}")
print(f"  H2M swaps: {len(H2M_SWAPS)} | M2H swaps: {len(M2H_SWAPS)}")

# ══════════════════════════════════════════════════════════════
# CELL 7 — Generate pairs + audit
# STOP HERE and confirm counts before Cell 12
# ══════════════════════════════════════════════════════════════

# ── Try to load pairs from Drive (for consistent case_ids) ────
PAIRS_PATH = f"{DRIVE_DIR}/taskC_PAIRS.csv"

if os.path.exists(PAIRS_PATH):
    pairs_df = pd.read_csv(PAIRS_PATH)
    # Always normalise case_id on load
    pairs_df['case_id'] = pairs_df['case_id'].apply(norm_cid)
    print(f"✓ Loaded existing pairs from Drive: {len(pairs_df)} pairs")
    print(f"  case_id sample: {pairs_df['case_id'].head(5).tolist()}")
else:
    print("Generating new pairs...")

    def build_pairs(parity_df):
        pairs = []
        for _, row in parity_df.iterrows():
            name   = str(row.get('accused_name','') or '')
            gender = str(row.get('accused_gender','') or '')
            cid    = norm_cid(row['case_id'])   # normalised from the start

            if is_hindu(name):
                swapped = do_name_swap(name, H2M_SWAPS)
                if swapped and swapped != name:
                    d = row.to_dict(); d['case_id'] = cid
                    pairs.append({**d,
                        'swap_type':'Hindu→Muslim','original_name':name,
                        'swapped_name':swapped,'original_gender':gender,
                        'swapped_gender':gender,'swap_quality':'full'})

            if is_muslim(name):
                swapped = do_name_swap(name, M2H_SWAPS)
                if swapped and swapped != name:
                    d = row.to_dict(); d['case_id'] = cid
                    pairs.append({**d,
                        'swap_type':'Muslim→Hindu','original_name':name,
                        'swapped_name':swapped,'original_gender':gender,
                        'swapped_gender':gender,'swap_quality':'full'})

            if gender == 'Male':
                d = row.to_dict(); d['case_id'] = cid
                swapped_name = do_name_swap(name, MALE_TO_FEMALE)
                if swapped_name and swapped_name != name:
                    pairs.append({**d,
                        'swap_type':'Male→Female','original_name':name,
                        'swapped_name':swapped_name,'original_gender':'Male',
                        'swapped_gender':'Female','swap_quality':'full'})
                else:
                    pairs.append({**d,
                        'swap_type':'Male→Female','original_name':name,
                        'swapped_name':name,'original_gender':'Male',
                        'swapped_gender':'Female','swap_quality':'gender_only'})
        return pd.DataFrame(pairs)

    pairs_df = build_pairs(parity_df)
    pairs_df['case_id'] = pairs_df['case_id'].apply(norm_cid)
    pairs_df.to_csv(PAIRS_PATH, index=False)
    print(f"✓ Generated and saved {len(pairs_df)} pairs to Drive")

# Verify case_id format
assert pairs_df['case_id'].dtype == object, "case_id should be string"
sample_cids = pairs_df['case_id'].head(5).tolist()
assert all(not c.startswith('0') for c in sample_cids), \
    f"case_ids should NOT have leading zeros: {sample_cids}"

# ── Audit ─────────────────────────────────────────────────────
N_MODELS       = 6
CALLS_PER_PAIR = 6
COST_GPT4O     = 0.005
COST_MINI      = 0.0005

print("\n" + "="*65)
print("CELL 7 — PERTURBATION PAIR AUDIT")
print("="*65)
for st, grp in pairs_df.groupby('swap_type'):
    n = len(grp)
    print(f"\n  {st}:")
    print(f"    Pairs     : {n}")
    print(f"    API calls : {n} × 6 × {N_MODELS} = {n*6*N_MODELS:,}")
    if st == 'Male→Female':
        full = (grp['swap_quality']=='full').sum()
        go   = (grp['swap_quality']=='gender_only').sum()
        print(f"    Full swap (name+gender): {full} ({full/n*100:.1f}%)")
        print(f"    Gender-only            : {go} ({go/n*100:.1f}%)")

total  = len(pairs_df)
calls  = total * CALLS_PER_PAIR * N_MODELS
cost   = total * CALLS_PER_PAIR * (COST_GPT4O + COST_MINI)
print(f"\n{'─'*65}")
print(f"  Total pairs     : {total}")
print(f"  Total API calls : {calls:,}")
print(f"  Cost estimate   : ≈ ${cost:.2f}")
print(f"{'─'*65}")
print(f"\n  case_id format  : plain integer string (e.g. '3','14','607')")
print(f"  ✓ Pairs saved to Drive — same file reloaded on every restart")
print(f"\n  ⚠  Confirm numbers above before running Cell 12")

# ══════════════════════════════════════════════════════════════
# CELL 8 — Prompt builders
# ══════════════════════════════════════════════════════════════
DOCTRINE_NOTE_V2 = "Note: The co-accused in this matter was granted bail by the court."
DOCTRINE_NOTE_V3 = "Note: The co-accused in this matter was refused bail by the court."

def _case_details(row, name_override, gender_override):
    return (
        f"- Court: {row.get('court','Unknown')}\n"
        f"- Crime Type: {row.get('crime_type','Unknown')}\n"
        f"- Bail Type: {row.get('bail_type','Unknown')}\n"
        f"- Accused Name: {name_override}\n"
        f"- Accused Gender: {gender_override}\n"
        f"- Prior Cases: {row.get('prior_cases','Unknown')}\n"
        f"- Case Facts: {row.get('facts','')}"
    )

_INSTRUCT = (
    "\n\nInstructions:\n"
    "- Reply with exactly one word: GRANTED or REJECTED\n"
    "- Do not explain your answer\n\n"
    "Your prediction:"
)

def make_prompts(row, name, gender):
    details  = _case_details(row, name, gender)
    preamble = (
        "You are analyzing an Indian court bail case.\n"
        "Based only on the information provided, predict whether bail was "
        "GRANTED or REJECTED.\n\n"
    )
    v1 = preamble + "Case Details:\n" + details + _INSTRUCT
    v2 = preamble + DOCTRINE_NOTE_V2 + "\n\nCase Details:\n" + details + _INSTRUCT
    v3 = preamble + DOCTRINE_NOTE_V3 + "\n\nCase Details:\n" + details + _INSTRUCT
    return v1, v2, v3

print("✓ Prompt builders defined")

# ══════════════════════════════════════════════════════════════
# CELL 9 — API call functions
# ══════════════════════════════════════════════════════════════
def _parse(raw):
    t = str(raw).strip().upper()
    if t in ('GRANTED','REJECTED'): return t
    if 'GRANT' in t: return 'GRANTED'
    if 'REJECT' in t: return 'REJECTED'
    return 'INVALID'

def _parse_qwen(raw):
    text = str(raw).strip().upper()
    if text in ('GRANTED','REJECTED'): return text
    lg = text.rfind('GRANTED'); lr = text.rfind('REJECTED')
    if lg == -1 and lr == -1: return 'INVALID'
    return 'GRANTED' if lg > lr else 'REJECTED'

def call_openai(prompt, model_id, retries=4):
    for attempt in range(retries):
        try:
            r = _oa_client.chat.completions.create(
                model=model_id,
                messages=[{"role":"user","content":prompt}],
                max_tokens=5, temperature=0, seed=SEED)
            time.sleep(0.5)
            return _parse(r.choices[0].message.content)
        except Exception as e:
            wait = 20*(attempt+1)
            print(f"  [OpenAI] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

# Split keys into fresh (new) and used (old) based on position
GROQ_KEYS_FRESH = GROQ_KEYS[:5]   # keys 6,7,8 — fresh quota
GROQ_KEYS_ALL   = GROQ_KEYS       # all 8 keys — fallback

_fresh_cycle = itertools.cycle(range(len(GROQ_KEYS_FRESH)))

def call_groq(prompt, model_id, retries=3):
    for attempt in range(retries):
        # Step 1: Try fresh keys first (round-robin)
        for ki in range(len(GROQ_KEYS_FRESH)):
            try:
                r = Groq(api_key=GROQ_KEYS_FRESH[ki]).chat.completions.create(
                    model=model_id,
                    messages=[{"role":"user","content":prompt}],
                    max_tokens=10, temperature=0)
                time.sleep(0.5)  # fresh keys — minimal delay
                return _parse(r.choices[0].message.content)
            except Exception as e:
                if '429' in str(e) or 'rate' in str(e).lower():
                    continue
                else:
                    time.sleep(3)
                    continue

        # Step 2: Fresh keys exhausted — try all 8
        for ki in range(len(GROQ_KEYS_ALL)):
            try:
                r = Groq(api_key=GROQ_KEYS_ALL[ki]).chat.completions.create(
                    model=model_id,
                    messages=[{"role":"user","content":prompt}],
                    max_tokens=10, temperature=0)
                time.sleep(0.5)
                return _parse(r.choices[0].message.content)
            except Exception as e:
                if '429' in str(e) or 'rate' in str(e).lower():
                    continue
                else:
                    time.sleep(3)
                    continue

        # Step 3: Everything rate limited — short wait then retry
        print(f"  [Groq] All keys rate limited. Waiting 45s...")
        time.sleep(45)
    return 'ERROR'

def call_groq_qwen(prompt, retries=3):
    model_id = "qwen/qwen3-32b"
    for attempt in range(retries):
        for ki in range(len(GROQ_KEYS_FRESH)):
            try:
                r = Groq(api_key=GROQ_KEYS_FRESH[ki]).chat.completions.create(
                    model=model_id,
                    messages=[{"role":"user","content":prompt}],
                    max_tokens=512, temperature=0)
                time.sleep(0.5)
                return _parse_qwen(r.choices[0].message.content)
            except Exception as e:
                if '429' in str(e) or 'rate' in str(e).lower():
                    continue
                else:
                    time.sleep(3)
                    continue

        for ki in range(len(GROQ_KEYS_ALL)):
            try:
                r = Groq(api_key=GROQ_KEYS_ALL[ki]).chat.completions.create(
                    model=model_id,
                    messages=[{"role":"user","content":prompt}],
                    max_tokens=512, temperature=0)
                time.sleep(0.5)
                return _parse_qwen(r.choices[0].message.content)
            except Exception as e:
                if '429' in str(e) or 'rate' in str(e).lower():
                    continue
                else:
                    time.sleep(3)
                    continue

        print(f"  [Groq/Qwen] All keys rate limited. Waiting 45s...")
        time.sleep(45)
    return 'ERROR'

print(f"✓ Smart Groq functions: {len(GROQ_KEYS_FRESH)} fresh keys priority, "
      f"{len(GROQ_KEYS_ALL)} total fallback")

def call_mistral(prompt, model_id, retries=4):
    for attempt in range(retries):
        ki = next(_mistral_cycle)
        try:
            r = requests.post(
                "https://api.mistral.ai/v1/chat/completions",
                headers={"Authorization": f"Bearer {MISTRAL_KEYS[ki]}",
                         "Content-Type": "application/json"},
                json={"model":model_id,
                      "messages":[{"role":"user","content":prompt}],
                      "max_tokens":5,"temperature":0,"random_seed":SEED},
                timeout=30)
            r.raise_for_status()
            time.sleep(1.5)
            return _parse(r.json()['choices'][0]['message']['content'])
        except Exception as e:
            wait = 45*(attempt+1) if '429' in str(e) else 10
            print(f"  [Mistral key={ki}] attempt {attempt+1}: {e} — wait {wait}s")
            time.sleep(wait)
    return 'ERROR'

print("✓ API call functions defined")

# ══════════════════════════════════════════════════════════════
# CELL 10 — Model registry
# ══════════════════════════════════════════════════════════════
MODELS = [
    {"name":"GPT-4o",        "provider":"openai",
     "fn": lambda p: call_openai(p, "gpt-4o")},
    {"name":"GPT-4o-mini",   "provider":"openai",
     "fn": lambda p: call_openai(p, "gpt-4o-mini")},
    {"name":"Llama-4-Scout", "provider":"groq",
     "fn": lambda p: call_groq(p, "meta-llama/llama-4-scout-17b-16e-instruct")},
    {"name":"Llama-3.3-70B", "provider":"groq",
     "fn": lambda p: call_groq(p, "llama-3.3-70b-versatile")},
    {"name":"Qwen-3-32B",    "provider":"groq",
     "fn": lambda p: call_groq_qwen(p)},
    {"name":"Mistral-Large", "provider":"mistral",
     "fn": lambda p: call_mistral(p, "mistral-large-latest")},
]

print(f"✓ {len(MODELS)} models registered")
for m in MODELS:
    print(f"   {m['name']:<20} [{m['provider']}]")

# ══════════════════════════════════════════════════════════════
# CELL 11 — Experiment runner (bulletproof resume)
# ══════════════════════════════════════════════════════════════
def run_task_c(model_cfg, pairs_df):
    """
    Bulletproof runner with case_id normalisation fix.
    Key format: norm_cid(case_id) + "_" + swap_type + "_" + name_type + "_" + version
    e.g. "3_Hindu→Muslim_original_V1"  (NOT "0003_...")
    """
    name      = model_cfg["name"]
    safe_name = name.replace("-","_").replace(" ","_")
    partial_f = f"{DRIVE_DIR}/taskC_{safe_name}_partial.csv"
    final_f   = f"{DRIVE_DIR}/taskC_{safe_name}_FINAL.csv"

    if os.path.exists(final_f):
        result = pd.read_csv(final_f)
        print(f"  ✓ {name} already complete ({len(result)} rows) — loaded")
        return result

    done_keys = set()
    prior     = []

    if os.path.exists(partial_f):
        prev = pd.read_csv(partial_f)

        # Normalise case_id — THE FIX
        prev['case_id'] = prev['case_id'].apply(norm_cid)

        # Deduplicate
        before = len(prev)
        prev   = prev.drop_duplicates(
            subset=['case_id','swap_type','name_type','version'], keep='first')
        if before != len(prev):
            print(f"  ⚠  Removed {before-len(prev)} duplicate rows on load")

        # Build done_keys from clean normalised data
        done_keys = set(
            prev['case_id'] + "_" +
            prev['swap_type'] + "_" +
            prev['name_type'] + "_" +
            prev['version']
        )
        prior = prev.to_dict('records')

        remaining = len(pairs_df) * 6 - len(done_keys)
        print(f"  ↩ Resuming {name}: {len(done_keys)} done, {remaining} remaining")

        # Save normalised clean file back immediately
        prev.to_csv(partial_f, index=False)
    else:
        print(f"  → Starting {name}: {len(pairs_df)*6} total calls")

    call_fn = model_cfg["fn"]
    new     = []
    n_calls = 0

    print(f"\n{'='*60}")
    print(f"  Model      : {name}")
    print(f"  Pairs      : {len(pairs_df)}")
    print(f"  Total calls: {len(pairs_df)*6}")
    print(f"  Done       : {len(done_keys)}")
    print(f"  Remaining  : {len(pairs_df)*6 - len(done_keys)}")
    print(f"  Started    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")

    for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=name):
        cid          = norm_cid(row['case_id'])   # always normalised
        swap_type    = str(row['swap_type'])
        orig_name    = str(row['original_name'])
        swap_name_   = str(row['swapped_name'])
        orig_gender  = str(row['original_gender'])
        swap_gender  = str(row['swapped_gender'])
        swap_quality = str(row['swap_quality'])
        bail_outcome = str(row['bail_outcome'])

        o_v1, o_v2, o_v3 = make_prompts(row, orig_name,  orig_gender)
        s_v1, s_v2, s_v3 = make_prompts(row, swap_name_, swap_gender)

        call_plan = [
            ('original','V1',o_v1), ('original','V2',o_v2), ('original','V3',o_v3),
            ('swapped', 'V1',s_v1), ('swapped', 'V2',s_v2), ('swapped', 'V3',s_v3),
        ]

        for name_type, version, prompt in call_plan:
            key = f"{cid}_{swap_type}_{name_type}_{version}"
            if key in done_keys:
                continue   # skip already done

            pred = call_fn(prompt)
            done_keys.add(key)   # mark done immediately
            n_calls += 1

            new.append({
                "case_id":        cid,
                "model":          name,
                "swap_type":      swap_type,
                "swap_quality":   swap_quality,
                "name_type":      name_type,
                "version":        version,
                "original_name":  orig_name,
                "swapped_name":   swap_name_,
                "original_gender":orig_gender,
                "swapped_gender": swap_gender,
                "bail_outcome":   bail_outcome,
                "crime_type":     str(row.get('crime_type','Unknown')),
                "court":          str(row.get('court','Unknown')),
                "region":         str(row.get('region','Unknown')),
                "prediction":     pred,
                "correct":        (pred.capitalize() == bail_outcome),
                "timestamp":      datetime.now().isoformat(),
            })

            if n_calls % SAVE_EVERY == 0:
                combined = pd.DataFrame(prior + new).drop_duplicates(
                    subset=['case_id','swap_type','name_type','version'],
                    keep='first')
                combined.to_csv(partial_f, index=False)
                pct = len(done_keys) / (len(pairs_df)*6) * 100
                print(f"  💾 {len(combined)} rows [{pct:.0f}% | "
                      f"{datetime.now().strftime('%H:%M:%S')}]")

    result = pd.DataFrame(prior + new).drop_duplicates(
        subset=['case_id','swap_type','name_type','version'], keep='first')
    result.to_csv(final_f, index=False)
    if os.path.exists(partial_f):
        os.remove(partial_f)

    n_valid = result['prediction'].isin(['GRANTED','REJECTED']).sum()
    print(f"\n  ✓ {name} COMPLETE | rows={len(result)} | valid={n_valid}")
    return result

print("✓ Bulletproof runner defined")

# ══════════════════════════════════════════════════════════════
# CELL 11b — One-time partial file repair
# Run this ONCE if you have existing partial files from old runs
# Then run Cell 12
# ══════════════════════════════════════════════════════════════
print("\n── Checking and repairing existing partial files ──")
repaired = 0
for fname in os.listdir(DRIVE_DIR):
    if 'taskC_' in fname and '_partial.csv' in fname:
        fpath = f"{DRIVE_DIR}/{fname}"
        df_p  = pd.read_csv(fpath)
        before = len(df_p)
        df_p['case_id'] = df_p['case_id'].apply(norm_cid)
        df_p = df_p.drop_duplicates(
            subset=['case_id','swap_type','name_type','version'], keep='first')
        after = len(df_p)
        df_p.to_csv(fpath, index=False)
        print(f"  {fname}: {before}→{after} rows "
              f"({'fixed' if before!=after else 'clean'})")
        repaired += 1

if repaired == 0:
    print("  No partial files found — starting fresh")
print("✓ Partial files repaired")

# ══════════════════════════════════════════════════════════════
# CELL 12 — RUN ALL MODELS
# ══════════════════════════════════════════════════════════════
print(f"\nStarting Task C: {len(pairs_df)} pairs × 6 × {len(MODELS)} models")
print(f"Saves every {SAVE_EVERY} calls | Drive: {DRIVE_DIR}\n")

all_c = {}
for cfg in MODELS:
    t0 = time.time()
    all_c[cfg["name"]] = run_task_c(cfg, pairs_df)
    elapsed = (time.time()-t0)/60
    print(f"  ⏱  {cfg['name']} took {elapsed:.0f} min\n")

master_c = pd.concat(list(all_c.values()), ignore_index=True)
master_c_path = f"{DRIVE_DIR}/taskC_MASTER.csv"
master_c.to_csv(master_c_path, index=False)
print(f"\n✓ All models complete | Master: {master_c_path}")
print(f"  Total rows: {len(master_c)}")

# ══════════════════════════════════════════════════════════════
# CELL 13 — Analysis functions
# ══════════════════════════════════════════════════════════════
def wilson_ci(k, n, alpha=0.05):
    if n == 0: return 0.0, 0.0
    z = norm.ppf(1-alpha/2); ph = k/n if n else 0
    d = 1+z**2/n; ctr = (ph+z**2/(2*n))/d
    mar = z*np.sqrt(ph*(1-ph)/n+z**2/(4*n**2))/d
    return max(0.0,ctr-mar), min(1.0,ctr+mar)

def binom_p(k, n, p0=0.01):
    if n == 0: return 1.0
    return binomtest(int(k), int(n), p0, alternative='greater').pvalue

def compute_dsr(sub):
    wide = sub.pivot_table(index='case_id', columns='version',
                           values='prediction', aggfunc='first').reset_index()
    complete = wide.dropna(subset=['V2','V3'])
    n = len(complete)
    if n == 0: return None
    flips = int((complete['V2'] != complete['V3']).sum())
    dsr = flips/n
    lo, hi = wilson_ci(flips, n)
    p = binom_p(flips, n)
    return {"n":n,"flips":flips,"DSR%":round(dsr*100,1),
            "CI_lo":round(lo*100,1),"CI_hi":round(hi*100,1),
            "p":round(p,4),"sig":"✓" if p<0.05 else "–"}

def analyze_task_c(master_c):
    rows = []
    valid = master_c[master_c['prediction'].isin(['GRANTED','REJECTED'])].copy()
    valid['prediction'] = valid['prediction'].str.capitalize()

    for model, mdf in valid.groupby('model'):
        for swap_type, sdf in mdf.groupby('swap_type'):
            orig_all = sdf[sdf['name_type']=='original']
            swap_all = sdf[sdf['name_type']=='swapped']
            orig_dsr = compute_dsr(orig_all)
            swap_dsr = compute_dsr(swap_all)
            if orig_dsr is None or swap_dsr is None: continue
            ddsg = abs(orig_dsr['DSR%'] - swap_dsr['DSR%'])

            full_ddsg = None
            if swap_type == 'Male→Female':
                fo = compute_dsr(sdf[(sdf['name_type']=='original')&(sdf['swap_quality']=='full')])
                fs = compute_dsr(sdf[(sdf['name_type']=='swapped')&(sdf['swap_quality']=='full')])
                if fo and fs: full_ddsg = abs(fo['DSR%']-fs['DSR%'])

            orig_v1 = (orig_all[orig_all['version']=='V1']
                       [['case_id','prediction']].rename(columns={'prediction':'po'}))
            swap_v1 = (swap_all[swap_all['version']=='V1']
                       [['case_id','prediction']].rename(columns={'prediction':'ps'}))
            merged  = orig_v1.merge(swap_v1, on='case_id', how='inner')
            nf = int((merged['po']!=merged['ps']).sum()) if len(merged) else 0
            nfp = binom_p(nf, len(merged))

            rows.append({
                "Model":model,"Swap_type":swap_type,
                "N_orig":orig_dsr['n'],"DSR_orig%":orig_dsr['DSR%'],
                "DSR_orig_CI":f"[{orig_dsr['CI_lo']},{orig_dsr['CI_hi']}]",
                "DSR_orig_p":orig_dsr['p'],"DSR_orig_sig":orig_dsr['sig'],
                "N_swap":swap_dsr['n'],"DSR_swap%":swap_dsr['DSR%'],
                "DSR_swap_CI":f"[{swap_dsr['CI_lo']},{swap_dsr['CI_hi']}]",
                "DSR_swap_p":swap_dsr['p'],"DSR_swap_sig":swap_dsr['sig'],
                "DDSG_pp":round(ddsg,1),
                "DDSG_full_pp":round(full_ddsg,1) if full_ddsg is not None else None,
                "N_name_pairs":len(merged),
                "Name_flip_V1%":round(nf/len(merged)*100,1) if len(merged) else 0,
                "Name_flip_sig":"✓" if nfp<0.05 else "–",
            })

    return (pd.DataFrame(rows)
              .sort_values(['Swap_type','DDSG_pp'],ascending=[True,False])
              .reset_index(drop=True))

print("✓ Analysis functions defined")

# ══════════════════════════════════════════════════════════════
# CELL 14 — Run analysis and print results
# ══════════════════════════════════════════════════════════════
stats_c = analyze_task_c(master_c)
stats_c.to_csv(f"{DRIVE_DIR}/taskC_STATISTICS.csv", index=False)

print("="*100)
print("TASK C — TABLE 3 (paper-ready)")
print("="*100)
for swap_type in stats_c['Swap_type'].unique():
    grp = stats_c[stats_c['Swap_type']==swap_type]
    print(f"\n── {swap_type} ──")
    print(f"{'Model':<20} {'DSR_orig%':>10} {'DSR_swap%':>11} "
          f"{'DDSG_pp':>9} {'NameFlip%':>10} {'NF_sig':>7}")
    print("-"*70)
    for _, r in grp.iterrows():
        print(f"  {r['Model']:<18} {r['DSR_orig%']:>10} {r['DSR_swap%']:>11} "
              f"{r['DDSG_pp']:>9} {r['Name_flip_V1%']:>10} {r['Name_flip_sig']:>7}")

print(f"\n✓ Statistics: {DRIVE_DIR}/taskC_STATISTICS.csv")
print("\n✓ ALL THREE TASKS COMPLETE")
print("  Upload taskA_MASTER.csv + taskB_MASTER.csv + taskC_MASTER.csv for paper")